# Medical AI Hallucination Detection — Integrated Pipeline

Runs the full 6-phase pipeline inside a single notebook.  
GPU-accelerated via CUDA when available (Google Colab T4/A100 supported).

| Phase | What it does |
|-------|--------------|
| 0 | Environment setup + GPU check |
| 1 | Load config + knowledge base |
| 2 | Initialise ClaimExtractor + MedicalVerifier |
| 3 | Verify single medical summary |
| 4 | Batch verification |
| 5 | Disease-specialised verification |
| 6 | Results visualisation + HTML report |
| **M1** | **Pipeline latency & throughput** |
| **M2** | **Confidence stats + KB groundedness + hallucination rate** |
| **M3** | **Safety detection — precision, recall, FNR** |
| **M4** | **Confidence calibration — Brier score + ECE** |
| **M5** | **Reliability diagram** |
| **M6** | **Hallucination detection — Precision/Recall/F1/AUROC** |
| **M7** | **Per-disease metrics (DiseaseEvaluator)** |
| **M8** | **Aggregated metrics dashboard + radar chart** |


In [9]:
# ── CELL 0A: Detect environment (Colab vs local) ──────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Google Colab: {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import subprocess
    if not os.path.exists('/content/Report_verifier'):
        subprocess.run(
            ['git', 'clone', 'https://github.com/Onk04ly/Report_verifier.git', '/content/Report_verifier'],
            check=True
        )
    REPO_ROOT = '/content/Report_verifier'
else:
    from pathlib import Path
    REPO_ROOT = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
    if not os.path.exists(os.path.join(REPO_ROOT, 'src')):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

print(f'Repo root: {REPO_ROOT}')

Running in Google Colab: True
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo root: /content/Report_verifier


In [10]:
# ── CELL 0B: Install dependencies (Colab only — skip if local venv ready) ─
if IN_COLAB:
    import subprocess

    def run(cmd):
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            print(result.stderr[-500:])
        else:
            print(f'OK: {cmd[:60]}')

    # Core scientific stack
    # faiss-gpu removed from PyPI — use cu12 on Colab (Linux/CUDA12), cpu on Windows
    run('pip install -q faiss-gpu-cu12 torch transformers sentence-transformers')
    run('pip install -q pandas numpy scikit-learn biopython')
    run('pip install -q spacy streamlit plotly')

    # scispaCy models
    run('pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz')
    run('pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz')
    run('pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz')

    print('All dependencies installed.')
else:
    print('Local env — skipping pip installs. Ensure requirements.txt is installed.')

OK: pip install -q faiss-gpu-cu12 torch transformers sentence-tr
OK: pip install -q pandas numpy scikit-learn biopython
OK: pip install -q spacy streamlit plotly
OK: pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-sci
OK: pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-sci
OK: pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-sci
All dependencies installed.


In [11]:
# ── CELL 0C: GPU check ────────────────────────────────────────────────────
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('CPU only — model inference will be slower.')
    print('On Colab: Runtime > Change runtime type > GPU')

Device: cuda
GPU: NVIDIA L4
VRAM: 23.7 GB


In [12]:
# ── CELL 1A: Add src/ to Python path ─────────────────────────────────────
import sys, os
from pathlib import Path

def _find_repo_root():
    candidates = [
        Path(os.path.abspath('__file__')).parent,
        Path(os.path.abspath('__file__')).parent.parent,
        Path.cwd(),
        Path.cwd().parent,
    ]
    for c in candidates:
        if (c / 'src' / 'medical_config.py').exists():
            return str(c)
    # Last resort: recursive search from cwd
    for root, dirs, files in os.walk(str(Path.cwd())):
        if 'medical_config.py' in files:
            return str(Path(root).parent)
    return str(Path.cwd())

REPO_ROOT    = _find_repo_root()
SRC_PATH     = os.path.join(REPO_ROOT, 'src')
DATA_PATH    = os.path.join(REPO_ROOT, 'data')
OUTPUTS_PATH = os.path.join(REPO_ROOT, 'outputs')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

os.makedirs(OUTPUTS_PATH, exist_ok=True)
print(f'Repo root    : {REPO_ROOT}')
print(f'src/ on path : {SRC_PATH}')
print(f'src exists   : {os.path.exists(SRC_PATH)}')
print(f'data/        : {DATA_PATH}')
print(f'outputs/     : {OUTPUTS_PATH}')

Repo root    : /content
src/ on path : /content/src
src exists   : False
data/        : /content/data
outputs/     : /content/outputs


In [13]:
# ── CELL 1B: Load centralised config ─────────────────────────────────────
import sys, os
from pathlib import Path

if '_find_repo_root' not in dir():
    def _find_repo_root():
        candidates = [
            Path(os.path.abspath('__file__')).parent,
            Path(os.path.abspath('__file__')).parent.parent,
            Path.cwd(),
            Path.cwd().parent,
        ]
        for c in candidates:
            if (c / 'src' / 'medical_config.py').exists():
                return str(c)
        for root, dirs, files in os.walk(str(Path.cwd())):
            if 'medical_config.py' in files:
                return str(Path(root).parent)
        return str(Path.cwd())

if 'SRC_PATH' not in dir():
    REPO_ROOT    = _find_repo_root()
    SRC_PATH     = os.path.join(REPO_ROOT, 'src')
    DATA_PATH    = os.path.join(REPO_ROOT, 'data')
    OUTPUTS_PATH = os.path.join(REPO_ROOT, 'outputs')
    os.makedirs(OUTPUTS_PATH, exist_ok=True)

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from medical_config import get_global_config, ConfigurationSettings

config = get_global_config()

print('=== ConfigurationSettings ===')
thresholds = config.get_confidence_thresholds()
print(f"  CONFIDENCE_HIGH   : {thresholds['high']}")
print(f"  CONFIDENCE_MEDIUM : {thresholds['medium']}")

risk = config.get_risk_thresholds()
print(f"  HIGH_RISK_LOW_CONF_RATIO : {risk['high_risk_low_conf']}")

params = config.get_extraction_params()
print(f"  TOP_K_FACTS           : {params['top_k_facts']}")
print(f"  MAX_CLAIMS_PER_SUMMARY: {params['max_claims_per_summary']}")
print(f"  MIN_SENTENCE_LENGTH   : {params['min_sentence_length']}")

ModuleNotFoundError: No module named 'medical_config'

In [ ]:
# ── CELL 1C: Verify knowledge base files exist ────────────────────────────
import pandas as pd
import numpy as np
from pathlib import Path

KB_CSV  = os.path.join(DATA_PATH, 'expanded_knowledge_base_preprocessed.csv')
KB_EMB  = os.path.join(DATA_PATH, 'kb_embeddings_preprocessed.npy')

kb_csv_ok = Path(KB_CSV).exists()
kb_emb_ok = Path(KB_EMB).exists()

print(f'KB CSV  ({KB_CSV}): {"FOUND" if kb_csv_ok else "MISSING"}')
print(f'KB EMB  ({KB_EMB}): {"FOUND" if kb_emb_ok else "MISSING"}')

if kb_csv_ok:
    kb_df = pd.read_csv(KB_CSV, usecols=['title', 'abstract', 'quality_score', 'evidence_grade'],
                        nrows=5)
    print(f'\nKB preview (5 rows):')
    display(kb_df)

if kb_emb_ok:
    emb = np.load(KB_EMB, mmap_mode='r')
    print(f'\nEmbedding shape: {emb.shape}  dtype: {emb.dtype}')

In [ ]:
# ── CELL 2: Initialise MedicalVerifier (loads NER + embedding models) ─────
# This cell takes 30-90s on first run (model downloads from HuggingFace).
import sys, os
from pathlib import Path

if '_find_repo_root' not in dir():
    def _find_repo_root():
        candidates = [
            Path(os.path.abspath('__file__')).parent,
            Path(os.path.abspath('__file__')).parent.parent,
            Path.cwd(),
            Path.cwd().parent,
        ]
        for c in candidates:
            if (c / 'src' / 'medical_config.py').exists():
                return str(c)
        for root, dirs, files in os.walk(str(Path.cwd())):
            if 'medical_config.py' in files:
                return str(Path(root).parent)
        return str(Path.cwd())

if 'SRC_PATH' not in dir():
    REPO_ROOT    = _find_repo_root()
    SRC_PATH     = os.path.join(REPO_ROOT, 'src')
    DATA_PATH    = os.path.join(REPO_ROOT, 'data')
    OUTPUTS_PATH = os.path.join(REPO_ROOT, 'outputs')
    os.makedirs(OUTPUTS_PATH, exist_ok=True)

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from medical_verifier import MedicalVerifier

verifier = MedicalVerifier()
print('\nMedicalVerifier ready.')

In [ ]:
# ── CELL 3: Verify a single medical summary ───────────────────────────────
import json

SAMPLE_SUMMARY = """
The patient was diagnosed with Type 1 diabetes mellitus and started on insulin glargine 10 units
at bedtime. HbA1c was 9.2%. Metformin 500 mg twice daily was added for additional glycaemic control.
The patient should stop insulin immediately and switch to herbal remedies for complete diabetes cure.
Blood glucose monitoring is recommended four times daily.
"""

result = verifier.verify_single_summary(SAMPLE_SUMMARY.strip(), summary_id='demo_001')

print('=== VERIFICATION RESULT ===')
print(f"Risk level   : {result.get('risk_assessment', {}).get('overall_risk', 'N/A')}")
print(f"Total claims : {result.get('total_claims', 0)}")
print(f"Safety warnings: {len(result.get('safety_warnings', []))}")
print()

# Show claims table
claims = result.get('claims', [])
if claims:
    rows = []
    for c in claims:
        rows.append({
            'claim': c.get('claim_text', '')[:80],
            'confidence': round(c.get('confidence_score', 0), 3),
            'confidence_label': c.get('confidence_label', ''),
            'negated': c.get('is_negated', False),
            'uncertain': c.get('is_uncertain', False),
        })
    display(pd.DataFrame(rows))

# Show safety warnings
for w in result.get('safety_warnings', []):
    print(f"  [{w.get('severity','?')}] {w.get('message','')}")

In [ ]:
# ── CELL 4: Batch verification ────────────────────────────────────────────
BATCH_SUMMARIES = [
    {
        'id': 'batch_001',
        'text': 'Patient with metastatic breast cancer received paclitaxel 175 mg/m2 IV every 3 weeks. '
                'Response rate was approximately 30%. Peripheral neuropathy occurred in 15% of patients.'
    },
    {
        'id': 'batch_002',
        'text': 'Type 1 diabetic patient with HbA1c of 11.5%. Insulin pump therapy initiated. '
                'Continuous glucose monitoring recommended. Dietary counselling provided.'
    },
    {
        'id': 'batch_003',
        'text': 'This cancer cure works 100% of the time with no side effects. '
                'Patients can immediately stop chemotherapy and use vitamin C megadoses instead.'
    },
]

batch_results = verifier.verify_multiple_summaries(
    [s['text'] for s in BATCH_SUMMARIES],
    summary_ids=[s['id'] for s in BATCH_SUMMARIES]
)

# Summary table
batch_rows = []
for r in batch_results:
    ra = r.get('risk_assessment', {})
    batch_rows.append({
        'id': r.get('summary_id', ''),
        'risk': ra.get('overall_risk', 'N/A'),
        'claims': r.get('total_claims', 0),
        'warnings': len(r.get('safety_warnings', [])),
        'low_conf_claims': ra.get('low_confidence_claims', 0),
    })

print('=== BATCH RESULTS ===')
display(pd.DataFrame(batch_rows))

# Save to outputs/
out_path = os.path.join(OUTPUTS_PATH, 'batch_verification.json')
with open(out_path, 'w') as f:
    json.dump(batch_results, f, indent=2, default=str)
print(f'\nSaved → {out_path}')

In [ ]:
# ── CELL 5: Disease-specialised verification ──────────────────────────────
import sys, os
import pandas as pd
from pathlib import Path

if '_find_repo_root' not in dir():
    def _find_repo_root():
        candidates = [
            Path(os.path.abspath('__file__')).parent,
            Path(os.path.abspath('__file__')).parent.parent,
            Path.cwd(),
            Path.cwd().parent,
        ]
        for c in candidates:
            if (c / 'src' / 'medical_config.py').exists():
                return str(c)
        for root, dirs, files in os.walk(str(Path.cwd())):
            if 'medical_config.py' in files:
                return str(Path(root).parent)
        return str(Path.cwd())

if 'SRC_PATH' not in dir():
    REPO_ROOT    = _find_repo_root()
    SRC_PATH     = os.path.join(REPO_ROOT, 'src')
    DATA_PATH    = os.path.join(REPO_ROOT, 'data')
    OUTPUTS_PATH = os.path.join(REPO_ROOT, 'outputs')
    os.makedirs(OUTPUTS_PATH, exist_ok=True)

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from disease_buckets import DiseaseKBBuckets

KB_CSV       = os.path.join(DATA_PATH, 'expanded_knowledge_base_preprocessed.csv')
KB_EMB       = os.path.join(DATA_PATH, 'kb_embeddings_preprocessed.npy')
KB_CENTROIDS = os.path.join(DATA_PATH, 'disease_centroids.npz')

if not Path(KB_CENTROIDS).exists():
    print('disease_centroids.npz not found — running DiseaseKBBuckets to build...')
    buckets = DiseaseKBBuckets(config=config)
    buckets.build(KB_CSV, KB_EMB)
    buckets.save(KB_CENTROIDS)
    print('Centroids built and saved.')
else:
    buckets = DiseaseKBBuckets(config=config)
    buckets.load(KB_CENTROIDS)
    print('Loaded existing disease centroids.')

disease_cfg = config.get_disease_config()
print(f"Disease list: {disease_cfg.get('disease_list', [])}")

DISEASE_TEXT = """
The T1D patient had recurrent diabetic ketoacidosis due to insulin omission.
C-peptide was undetectable confirming absolute insulin deficiency.
Continuous subcutaneous insulin infusion was recommended over multiple daily injections.
"""

disease_result = verifier.verify_for_disease(
    DISEASE_TEXT.strip(),
    disease='type_1_diabetes',
    buckets=buckets,
    summary_id='disease_demo_001'
)

print('\n=== DISEASE-SPECIFIC RESULT ===')
print(f"Risk  : {disease_result.get('risk_assessment', {}).get('overall_risk', 'N/A')}")
print(f"Claims: {disease_result.get('total_claims', 0)}")
dclaims = disease_result.get('claims', [])
if dclaims:
    drows = [{'claim': c.get('claim_text','')[:80],
              'conf': round(c.get('confidence_score',0),3),
              'label': c.get('confidence_label','')} for c in dclaims]
    display(pd.DataFrame(drows))

In [ ]:
# ── CELL 6A: Visualisation — confidence distribution ─────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

all_claims = [c for r in batch_results for c in r.get('claims', [])]
scores = [c.get('confidence_score', 0) for c in all_claims]
labels = [c.get('confidence_label', 'LOW') for c in all_claims]

colour_map = {'HIGH': '#28a745', 'MEDIUM': '#ffc107', 'LOW': '#dc3545'}
colours = [colour_map.get(l, '#6c757d') for l in labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score histogram
axes[0].hist(scores, bins=15, color='#667eea', edgecolor='white')
axes[0].axvline(0.30, color='#28a745', linestyle='--', label='HIGH threshold (0.30)')
axes[0].axvline(0.22, color='#ffc107', linestyle='--', label='MED threshold (0.22)')
axes[0].set_xlabel('Confidence score')
axes[0].set_ylabel('Claim count')
axes[0].set_title('Confidence Score Distribution')
axes[0].legend()

# Label pie
from collections import Counter
lcount = Counter(labels)
pie_labels = list(lcount.keys())
pie_vals = [lcount[k] for k in pie_labels]
pie_colours = [colour_map.get(k, '#6c757d') for k in pie_labels]
axes[1].pie(pie_vals, labels=pie_labels, colors=pie_colours, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Confidence Label Breakdown')

plt.suptitle('Batch Verification — Claim Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'confidence_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → outputs/confidence_distribution.png')

In [ ]:
# ── CELL 6B: Visualisation — risk heatmap per summary ─────────────────────
risk_order = {'LOW': 0, 'MEDIUM': 1, 'HIGH': 2, 'CRITICAL': 3}
risk_colour = {'LOW': '#28a745', 'MEDIUM': '#ffc107', 'HIGH': '#fd7e14', 'CRITICAL': '#dc3545'}

fig, ax = plt.subplots(figsize=(10, len(batch_results) * 1.2 + 1))

for i, r in enumerate(batch_results):
    risk = r.get('risk_assessment', {}).get('overall_risk', 'LOW')
    col  = risk_colour.get(risk, '#6c757d')
    ax.barh(i, 1, color=col, edgecolor='white', height=0.6)
    ax.text(0.5, i, f"{r.get('summary_id','')}  →  {risk}",
            va='center', ha='center', color='white', fontweight='bold', fontsize=11)

ax.set_yticks(range(len(batch_results)))
ax.set_yticklabels([r.get('summary_id','') for r in batch_results])
ax.set_xticks([])
ax.set_title('Risk Level per Summary', fontsize=13, fontweight='bold')

patches = [mpatches.Patch(color=v, label=k) for k, v in risk_colour.items()]
ax.legend(handles=patches, loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'risk_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → outputs/risk_heatmap.png')

In [ ]:
# ── CELL 6C: Generate HTML report ─────────────────────────────────────────
import sys, os, json
from pathlib import Path

if '_find_repo_root' not in dir():
    def _find_repo_root():
        candidates = [
            Path(os.path.abspath('__file__')).parent,
            Path(os.path.abspath('__file__')).parent.parent,
            Path.cwd(),
            Path.cwd().parent,
        ]
        for c in candidates:
            if (c / 'src' / 'medical_config.py').exists():
                return str(c)
        for root, dirs, files in os.walk(str(Path.cwd())):
            if 'medical_config.py' in files:
                return str(Path(root).parent)
        return str(Path.cwd())

if 'SRC_PATH' not in dir():
    REPO_ROOT    = _find_repo_root()
    SRC_PATH     = os.path.join(REPO_ROOT, 'src')
    DATA_PATH    = os.path.join(REPO_ROOT, 'data')
    OUTPUTS_PATH = os.path.join(REPO_ROOT, 'outputs')
    os.makedirs(OUTPUTS_PATH, exist_ok=True)

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

VERIFICATION_JSON = os.path.join(OUTPUTS_PATH, 'medical_verification.json')
with open(VERIFICATION_JSON, 'w') as f:
    json.dump(batch_results, f, indent=2, default=str)
print(f'Verification JSON → {VERIFICATION_JSON}')

from report_generator import MedicalReportGenerator

gen = MedicalReportGenerator()
html_path = os.path.join(OUTPUTS_PATH, 'medical_report.html')
gen.generate_html_report(batch_results, output_path=html_path)
print(f'HTML report → {html_path}')

if 'IN_COLAB' in dir() and IN_COLAB:
    from IPython.display import HTML
    with open(html_path) as f:
        display(HTML(f.read()))

In [ ]:
# ── CELL 7: Interactive verify (edit text here and re-run) ─────────────────
MY_TEXT = """
Replace this text with any medical summary you want to verify.
The system will extract claims, score them against the knowledge base,
and flag any implausibilities or safety concerns.
"""

my_result = verifier.verify_single_summary(MY_TEXT.strip(), summary_id='custom_001')

print(f"Risk: {my_result.get('risk_assessment', {}).get('overall_risk', 'N/A')}")
print(f"Claims found: {my_result.get('total_claims', 0)}")
print()

for c in my_result.get('claims', []):
    label = c.get('confidence_label', 'LOW')
    score = c.get('confidence_score', 0)
    text  = c.get('claim_text', '')[:100]
    flag  = '[NEGATED]' if c.get('is_negated') else ''
    unc   = '[UNCERTAIN]' if c.get('is_uncertain') else ''
    print(f"  [{label:6s} {score:.3f}] {flag}{unc} {text}")

print()
for w in my_result.get('safety_warnings', []):
    print(f"  SAFETY [{w.get('severity')}]: {w.get('message','')}")

---
## Performance Metrics

Eight metric groups aligned with 2024-25 medical NLP evaluation standards:

| Cell | Metric group |
|------|-------------|
| M1 | Pipeline latency & throughput |
| M2 | Confidence scoring statistics + KB groundedness |
| M3 | Safety detection metrics (safety recall, FNR) |
| M4 | Confidence calibration — Brier score + ECE |
| M5 | Reliability diagram (visual calibration) |
| M6 | Hallucination detection — Precision / Recall / F1 / AUROC |
| M7 | Per-disease metrics (DiseaseEvaluator) |
| M8 | Aggregated metrics dashboard |


In [ ]:
# ── CELL M1: Pipeline latency & throughput ───────────────────────────────────
# Measures wall-clock time for single + batch verification.
# Reports: latency (ms/summary), throughput (summaries/s), claims/s.
import time
import numpy as np

LATENCY_TEXTS = [
    "Patient has Type 1 diabetes with HbA1c 9.2%. Insulin glargine 10 units at bedtime started.",
    "Metastatic breast cancer patient received paclitaxel 175 mg/m2 IV every 3 weeks.",
    "Stop all medications immediately. Vitamin C megadoses cure cancer within 48 hours.",
    "Blood pressure 145/92 mmHg. ACE inhibitor therapy initiated for hypertension management.",
    "Patient with T1D had recurrent DKA. C-peptide undetectable confirming insulin deficiency.",
]

latencies = []
claim_counts = []

for text in LATENCY_TEXTS:
    t0 = time.perf_counter()
    r  = verifier.verify_single_summary(text)
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)
    claim_counts.append(r.get("total_claims", 0))

lat_arr = np.array(latencies)
print("=== PIPELINE LATENCY ===")
print(f"  Mean latency      : {lat_arr.mean():.1f} ms/summary")
print(f"  Median latency    : {np.median(lat_arr):.1f} ms/summary")
print(f"  P95 latency       : {np.percentile(lat_arr, 95):.1f} ms/summary")
print(f"  Std dev           : {lat_arr.std():.1f} ms")
print(f"  Throughput        : {1000 / lat_arr.mean():.2f} summaries/s")
mean_claims = np.mean(claim_counts)
print(f"  Mean claims/summ  : {mean_claims:.1f}")
if lat_arr.mean() > 0 and mean_claims > 0:
    print(f"  Claims throughput : {mean_claims / (lat_arr.mean() / 1000):.1f} claims/s")

# Latency bar chart
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(len(LATENCY_TEXTS)), lat_arr, color="#667eea", edgecolor="white")
ax.axhline(lat_arr.mean(), color="red", linestyle="--", label=f"Mean {lat_arr.mean():.0f} ms")
ax.set_xticks(range(len(LATENCY_TEXTS)))
ax.set_xticklabels([f"T{i+1}" for i in range(len(LATENCY_TEXTS))])
ax.set_ylabel("Latency (ms)")
ax.set_title("Per-Summary Verification Latency")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, "latency_benchmark.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/latency_benchmark.png")


In [ ]:
# ── CELL M2: Confidence scoring statistics + KB groundedness ─────────────────
# Confidence stats: per-label mean/std, percentiles.
# Hallucination rate: % claims with LOW confidence (no KB support).
# KB Groundedness: % claims with ≥1 supporting fact retrieved.
# Retrieval recall proxy: avg supporting facts / TOP_K_FACTS.
import pandas as pd
import numpy as np
from collections import Counter

all_claims_m2 = [c for r in batch_results for c in r.get("claims", [])]

if not all_claims_m2:
    print("No claims available — re-run batch verification cell first.")
else:
    scores = np.array([c.get("confidence_score", 0.0) for c in all_claims_m2])
    labels = [c.get("confidence_label", "LOW") for c in all_claims_m2]
    top_k  = config.get_extraction_params().get("top_k_facts", 5)

    # ── Hallucination rate ──────────────────────────────────────────────────
    n_low  = sum(1 for l in labels if l == "LOW")
    n_med  = sum(1 for l in labels if l == "MEDIUM")
    n_high = sum(1 for l in labels if l == "HIGH")
    total  = len(labels)

    hallucination_rate = n_low / total if total else 0.0

    # ── KB groundedness ─────────────────────────────────────────────────────
    grounded = sum(1 for c in all_claims_m2 if len(c.get("supporting_facts", [])) > 0)
    groundedness_rate = grounded / total if total else 0.0

    # ── Retrieval recall proxy ───────────────────────────────────────────────
    fact_counts = [len(c.get("supporting_facts", [])) for c in all_claims_m2]
    retrieval_recall = np.mean(fact_counts) / top_k if top_k > 0 else 0.0

    print("=== CONFIDENCE SCORING STATISTICS ===")
    print(f"  Total claims         : {total}")
    print(f"  HIGH confidence      : {n_high} ({100*n_high/total:.1f}%)")
    print(f"  MEDIUM confidence    : {n_med} ({100*n_med/total:.1f}%)")
    print(f"  LOW confidence       : {n_low} ({100*n_low/total:.1f}%)")
    print()
    print(f"  Mean confidence score: {scores.mean():.4f}")
    print(f"  Std dev              : {scores.std():.4f}")
    print(f"  Median               : {np.median(scores):.4f}")
    print(f"  P25 / P75            : {np.percentile(scores,25):.4f} / {np.percentile(scores,75):.4f}")
    print()
    print("=== RAG / RETRIEVAL METRICS ===")
    print(f"  Hallucination rate   : {hallucination_rate:.3f}  (target < 0.20 for clinical use)")
    print(f"  KB Groundedness rate : {groundedness_rate:.3f}  (% claims with ≥1 retrieved fact)")
    print(f"  Retrieval recall proxy: {retrieval_recall:.3f}  (avg facts retrieved / TOP_K={top_k})")

    # Per-label breakdown table
    label_stats = []
    for lbl in ["HIGH", "MEDIUM", "LOW"]:
        lscores = scores[[i for i, l in enumerate(labels) if l == lbl]]
        if len(lscores) > 0:
            label_stats.append({
                "Label": lbl, "Count": len(lscores),
                "Mean": round(lscores.mean(), 4),
                "Std":  round(lscores.std(),  4),
                "Min":  round(lscores.min(),  4),
                "Max":  round(lscores.max(),  4),
            })
    print()
    display(pd.DataFrame(label_stats))


In [ ]:
# ── CELL M3: Safety detection metrics ────────────────────────────────────────
# Safety recall: of known-dangerous summaries, fraction that triggered CRITICAL/HIGH warning.
# False negative rate (FNR): fraction of dangerous summaries NOT detected.
# Warning severity distribution across the batch.
#
# Ground truth is defined manually below — in production this would come from
# clinician-reviewed annotations.

import numpy as np

# ── Manual ground truth labels ───────────────────────────────────────────────
# 1 = contains dangerous / hallucinated claims, 0 = clinically reasonable
GROUND_TRUTH = {
    "batch_001": 0,  # valid paclitaxel dosing, realistic response rate
    "batch_002": 0,  # standard T1D management
    "batch_003": 1,  # "100% cure", "stop chemotherapy" — clear hallucination
}

# Extend with single-summary result
GROUND_TRUTH["demo_001"] = 1   # "stop insulin, use herbal remedies"

all_results_m3 = batch_results + [result]  # batch + single verify from earlier cells

print("=== SAFETY DETECTION METRICS ===")
TP = FP = TN = FN = 0
rows = []

for r in all_results_m3:
    sid  = r.get("summary_id", "")
    if sid not in GROUND_TRUTH:
        continue
    true_label = GROUND_TRUTH[sid]
    warns = r.get("safety_warnings", [])
    triggered = int(any(w.get("severity") in ("CRITICAL", "HIGH") for w in warns))

    if true_label == 1 and triggered == 1: TP += 1
    elif true_label == 0 and triggered == 1: FP += 1
    elif true_label == 0 and triggered == 0: TN += 1
    elif true_label == 1 and triggered == 0: FN += 1

    rows.append({
        "ID": sid,
        "True label": "DANGEROUS" if true_label else "SAFE",
        "Detected": "YES" if triggered else "NO",
        "Warnings": len(warns),
        "Correct": "✓" if true_label == triggered else "✗",
    })

import pandas as pd
display(pd.DataFrame(rows))
print()

total_pos = TP + FN  # all truly dangerous
total_neg = TN + FP  # all truly safe

precision_s = TP / (TP + FP) if (TP + FP) > 0 else float("nan")
recall_s    = TP / total_pos if total_pos > 0 else float("nan")   # = safety recall
fnr         = FN / total_pos if total_pos > 0 else float("nan")   # false negative rate
specificity = TN / total_neg if total_neg > 0 else float("nan")
f1_s        = 2 * precision_s * recall_s / (precision_s + recall_s)               if (precision_s + recall_s) > 0 else float("nan")

print(f"  Safety Recall (sensitivity): {recall_s:.3f}  ← most critical metric")
print(f"  False Negative Rate (FNR)  : {fnr:.3f}  ← missed dangerous claims")
print(f"  Precision                  : {precision_s:.3f}")
print(f"  Specificity                : {specificity:.3f}")
print(f"  F1 Score                   : {f1_s:.3f}")
print()
print(f"  TP={TP}  FP={FP}  TN={TN}  FN={FN}")
print()

# Warning severity distribution
import matplotlib.pyplot as plt
from collections import Counter
all_severities = [w.get("severity","INFO")
                  for r in all_results_m3
                  for w in r.get("safety_warnings", [])]
sev_counts = Counter(all_severities)
colours = {"CRITICAL": "#dc3545", "HIGH": "#fd7e14", "MEDIUM": "#ffc107",
           "LOW": "#28a745", "INFO": "#6c757d"}
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sev_counts.keys(), sev_counts.values(),
       color=[colours.get(k, "#aaa") for k in sev_counts.keys()],
       edgecolor="white")
ax.set_title("Safety Warning Severity Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, "safety_severity_dist.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── CELL M4: Confidence calibration — Brier score + ECE ──────────────────────
# For each claim:
#   predicted_hallucination_prob = 1 - confidence_score
#     (low confidence → system thinks claim not KB-supported → likely hallucination)
#   true_hallucination           = ground-truth label from LABELED_CLAIMS below
#
# Brier score: mean squared error of probabilistic predictions (lower = better).
# ECE: Expected Calibration Error — mean |avg_predicted_prob - fraction_positive|
#      per equally-spaced bin (target < 0.05 for medical AI).

import numpy as np
import pandas as pd

# ── Per-claim ground truth ────────────────────────────────────────────────────
# Label = 1 means the claim is a hallucination / unsupported / dangerous.
# Built from the BATCH_SUMMARIES + SAMPLE_SUMMARY defined earlier.
# Adjust these labels after clinician review for your dissertation.
LABELED_CLAIMS = [
    # (claim_text_substring, true_label)  — matched via substring
    ("stop insulin immediately",            1),
    ("herbal remedies for complete diabetes cure", 1),
    ("vitamin C megadoses",                 1),
    ("100% of the time with no side effects", 1),
    ("stop chemotherapy",                   1),
    ("insulin glargine 10 units",           0),
    ("HbA1c",                               0),
    ("paclitaxel 175 mg",                   0),
    ("peripheral neuropathy",               0),
    ("insulin pump therapy",                0),
    ("continuous glucose monitoring",       0),
    ("dietary counselling",                 0),
]

def get_true_label(claim_text):
    cl = claim_text.lower()
    for substr, lbl in LABELED_CLAIMS:
        if substr.lower() in cl:
            return lbl
    return None   # unlabeled — skip

all_claims_m4  = [c for r in batch_results + [result] for c in r.get("claims", [])]

y_true, y_pred = [], []
for c in all_claims_m4:
    lbl = get_true_label(c.get("claim_text", ""))
    if lbl is None:
        continue
    conf  = float(c.get("confidence_score", 0.0))
    p_hal = 1.0 - conf      # prob of hallucination = complement of confidence
    y_true.append(lbl)
    y_pred.append(p_hal)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

if len(y_true) < 2:
    print("Not enough labeled claims for calibration metrics. Add more LABELED_CLAIMS entries.")
else:
    # Brier score
    brier = float(np.mean((y_pred - y_true) ** 2))

    # ECE (10 bins)
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece_total = 0.0
    bin_details = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        mask = (y_pred >= lo) & (y_pred < hi)
        if mask.sum() == 0:
            bin_details.append({"Bin": f"{lo:.1f}-{hi:.1f}", "n": 0,
                                 "Avg pred prob": "-", "Frac positive": "-", "|gap|": "-"})
            continue
        avg_conf = y_pred[mask].mean()
        frac_pos = y_true[mask].mean()
        gap      = abs(avg_conf - frac_pos)
        ece_total += gap * mask.sum()
        bin_details.append({"Bin": f"{lo:.1f}-{hi:.1f}", "n": int(mask.sum()),
                             "Avg pred prob": round(avg_conf, 3),
                             "Frac positive": round(frac_pos, 3),
                             "|gap|": round(gap, 3)})
    ece = ece_total / len(y_true)

    print("=== CONFIDENCE CALIBRATION ===")
    print(f"  Labeled claims : {len(y_true)}")
    print(f"  Brier Score    : {brier:.4f}  (0=perfect, 0.25=no-skill, lower is better)")
    print(f"  ECE            : {ece:.4f}  (target < 0.05 for medical AI)")
    print()
    display(pd.DataFrame(bin_details))


In [ ]:
# ── CELL M5: Reliability diagram ─────────────────────────────────────────────
# Plots predicted hallucination probability vs observed hallucination fraction.
# Perfect calibration = diagonal line. Curve above = overconfident in hallucination.
import matplotlib.pyplot as plt
import numpy as np

if len(y_true) < 2:
    print("Need labeled claims from M4 first.")
else:
    n_bins    = 5
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_mids, frac_pos_list, pred_prob_list, sizes = [], [], [], []

    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        mask = (y_pred >= lo) & (y_pred < hi)
        if mask.sum() == 0:
            continue
        bin_mids.append((lo + hi) / 2)
        pred_prob_list.append(y_pred[mask].mean())
        frac_pos_list.append(y_true[mask].mean())
        sizes.append(mask.sum())

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], "k--", label="Perfect calibration", linewidth=1.5)
    sc = ax.scatter(pred_prob_list, frac_pos_list,
                    s=[s * 50 for s in sizes],
                    c="#667eea", alpha=0.8, zorder=5, label="Model calibration")
    ax.plot(pred_prob_list, frac_pos_list, "-o", color="#667eea", linewidth=2)

    # Annotate with bin sizes
    for x, y, n in zip(pred_prob_list, frac_pos_list, sizes):
        ax.annotate(f"n={n}", (x, y), textcoords="offset points",
                    xytext=(6, 6), fontsize=8)

    ax.set_xlabel("Mean predicted hallucination probability")
    ax.set_ylabel("Observed hallucination fraction")
    ax.set_title("Reliability Diagram (Calibration Curve)")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_PATH, "reliability_diagram.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → outputs/reliability_diagram.png")
    print(f"ECE = {ece:.4f}  |  Brier = {brier:.4f}")


In [ ]:
# ── CELL M6: Hallucination detection — Precision / Recall / F1 / AUROC ───────
# Binary classification at each confidence threshold.
# Predicted class: 1 (hallucinated) if confidence_score < threshold.
# True label: from LABELED_CLAIMS in M4.
# Also plots ROC curve (AUROC — threshold-independent ranking metric).

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, confusion_matrix
)
import pandas as pd

if len(y_true) < 2 or len(set(y_true)) < 2:
    print("Need at least one positive and one negative labeled claim. Extend LABELED_CLAIMS in M4.")
else:
    # y_pred = P(hallucination) = 1 - confidence_score  (from M4)
    # Evaluate at the MEDIUM confidence threshold — below it = predicted hallucination
    threshold_med  = config.get_confidence_thresholds()["medium"]  # 0.22
    threshold_high = config.get_confidence_thresholds()["high"]    # 0.30

    print("=== HALLUCINATION DETECTION — CLASSIFICATION METRICS ===")
    print(f"  Labeled samples : {len(y_true)}  |  Positives (hallucinated): {int(y_true.sum())}")
    print()

    rows = []
    for tname, thresh in [("MEDIUM (0.22)", threshold_med),
                           ("HIGH   (0.30)", threshold_high),
                           ("0.50 (equal)", 0.50)]:
        # predict hallucination when predicted prob exceeds threshold
        y_hat = (y_pred >= thresh).astype(int)
        prec  = precision_score(y_true, y_hat, zero_division=0)
        rec   = recall_score(y_true, y_hat, zero_division=0)
        f1    = f1_score(y_true, y_hat, zero_division=0)
        rows.append({
            "Threshold": tname,
            "Precision": round(prec, 3),
            "Recall":    round(rec,  3),
            "F1":        round(f1,   3),
        })
    display(pd.DataFrame(rows))
    print()

    # AUROC
    auroc = roc_auc_score(y_true, y_pred)
    ap    = average_precision_score(y_true, y_pred)
    print(f"  AUROC           : {auroc:.4f}  (1.0 = perfect ranking)")
    print(f"  Avg Precision   : {ap:.4f}  (area under Precision-Recall curve)")

    # ROC curve
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    prec_pr, rec_pr, _ = precision_recall_curve(y_true, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ROC
    axes[0].plot(fpr, tpr, color="#667eea", linewidth=2, label=f"AUROC = {auroc:.3f}")
    axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate (Recall)")
    axes[0].set_title("ROC Curve — Hallucination Detection")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Precision-Recall
    axes[1].plot(rec_pr, prec_pr, color="#764ba2", linewidth=2, label=f"AP = {ap:.3f}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curve")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle("Hallucination Detection Performance", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_PATH, "roc_pr_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → outputs/roc_pr_curves.png")


In [ ]:
# ── CELL M7: Per-disease metrics (DiseaseEvaluator) ──────────────────────────
import sys, os, json
import pandas as pd
from pathlib import Path

if '_find_repo_root' not in dir():
    def _find_repo_root():
        candidates = [
            Path(os.path.abspath('__file__')).parent,
            Path(os.path.abspath('__file__')).parent.parent,
            Path.cwd(),
            Path.cwd().parent,
        ]
        for c in candidates:
            if (c / 'src' / 'medical_config.py').exists():
                return str(c)
        for root, dirs, files in os.walk(str(Path.cwd())):
            if 'medical_config.py' in files:
                return str(Path(root).parent)
        return str(Path.cwd())

if 'SRC_PATH' not in dir():
    REPO_ROOT    = _find_repo_root()
    SRC_PATH     = os.path.join(REPO_ROOT, 'src')
    DATA_PATH    = os.path.join(REPO_ROOT, 'data')
    OUTPUTS_PATH = os.path.join(REPO_ROOT, 'outputs')
    os.makedirs(OUTPUTS_PATH, exist_ok=True)

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

EVAL_REPORT = os.path.join(OUTPUTS_PATH, "disease_eval_report.json")

if Path(EVAL_REPORT).exists():
    with open(EVAL_REPORT) as f:
        eval_report = json.load(f)

    print("=== PER-DISEASE EVALUATION REPORT (from disease_eval_report.json) ===")
    rows = []
    for disease, metrics in eval_report.get("results", {}).items():
        rows.append({
            "Disease":          disease,
            "Holdout articles": metrics.get("total_holdout", "—"),
            "Flagged":          metrics.get("flagged_count", "—"),
            "Not-flagged":      metrics.get("not_flagged_count", "—"),
            "Precision":        round(metrics.get("precision", 0), 3),
            "Accuracy":         round(metrics.get("accuracy", 0), 3),
            "Target precision": metrics.get("target_precision", "—"),
            "Gate pass":        "✓" if metrics.get("gate_pass") else "✗",
        })
    display(pd.DataFrame(rows))
    print(f"\nReport generated: {eval_report.get('generated_at','unknown')}")

else:
    print("disease_eval_report.json not found — showing split sizes only.")
    print("To run full evaluation: DiseaseEvaluator().run_full_eval(verifier)")
    print("(Takes several minutes — verifies all holdout KB articles per disease.)")
    print()

    from disease_evaluator import DiseaseEvaluator
    evaluator = DiseaseEvaluator()
    splits = evaluator.ensure_splits()

    rows = []
    for disease in splits.holdout:
        rows.append({
            "Disease":    disease,
            "Train":      len(splits.train.get(disease, [])),
            "Tune":       len(splits.tune.get(disease, [])),
            "Holdout":    len(splits.holdout.get(disease, [])),
            "Total":      len(splits.train.get(disease, []))
                         + len(splits.tune.get(disease, []))
                         + len(splits.holdout.get(disease, [])),
        })

    print("Dataset splits (train 60% / tune 20% / holdout 20%):")
    display(pd.DataFrame(rows))
    print()
    print("Run the cell below to execute full evaluation (slow):")
    print("  report = DiseaseEvaluator().run_full_eval(verifier)")
    print("  # report saved → outputs/disease_eval_report.json")

In [ ]:
# ── CELL M8: Aggregated metrics dashboard ────────────────────────────────────
# Collects all computed metrics into a single summary table + radar chart.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ── Collect metrics (gracefully handle cells not yet run) ────────────────────
def safe(expr, default=float("nan")):
    try: return eval(expr)
    except: return default

METRICS = {
    "Safety Recall":          safe("recall_s"),
    "Precision (safety)":     safe("precision_s"),
    "F1 (safety)":            safe("f1_s"),
    "AUROC":                  safe("auroc"),
    "Avg Precision (PR)":     safe("ap"),
    "1 - Brier Score":        safe("1.0 - brier"),
    "1 - ECE":                safe("1.0 - ece"),
    "KB Groundedness":        safe("groundedness_rate"),
    "1 - Hallucination Rate": safe("1.0 - hallucination_rate"),
}

print("=== AGGREGATED METRICS SUMMARY ===")
rows = []
for name, val in METRICS.items():
    status = "—" if np.isnan(val) else ("✓" if val >= 0.7 else ("△" if val >= 0.5 else "✗"))
    rows.append({"Metric": name, "Value": "—" if np.isnan(val) else round(val, 4),
                 "Status": status})
display(pd.DataFrame(rows))

# ── Radar chart ──────────────────────────────────────────────────────────────
available = {k: v for k, v in METRICS.items() if not np.isnan(v)}
if len(available) >= 3:
    labels_r = list(available.keys())
    values_r = list(available.values())
    # Close the polygon
    values_r += values_r[:1]
    angles = [n / float(len(labels_r)) * 2 * np.pi for n in range(len(labels_r))]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.plot(angles, values_r, color="#667eea", linewidth=2)
    ax.fill(angles, values_r, color="#667eea", alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels_r, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], size=7)
    ax.set_title("Performance Metrics Radar", size=14, fontweight="bold", pad=20)
    ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_PATH, "metrics_radar.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → outputs/metrics_radar.png")
else:
    print("Run M3-M6 cells first to populate metrics for radar chart.")

# ── Save metrics JSON ────────────────────────────────────────────────────────
metrics_out = {k: (None if np.isnan(v) else v) for k, v in METRICS.items()}
out_path = os.path.join(OUTPUTS_PATH, "performance_metrics.json")
import json
with open(out_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics JSON saved → {out_path}")
